In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [28]:
train_df=pd.read_csv("D:\medical-11\data\diet_recommendations_cleaned.csv")

In [29]:
train_df.drop(columns=['Calculated_Calorie_Intake'], inplace=True)

In [30]:
train_df.drop(columns=["Patient_ID" , 'Daily_Caloric_Intake', 'Adherence_to_Diet_Plan', 'Dietary_Nutrient_Imbalance_Score','BMI','Preferred_Cuisine','Dietary_Restrictions'],inplace=True) 

In [31]:
train_df.sample(5)

,Age,Gender,Weight_kg,Height_cm,Disease_Type,Severity,Physical_Activity_Level,Cholesterol_mg/dL,Blood_Pressure_mmHg,Glucose_mg/dL,Allergies,Weekly_Exercise_Hours,Diet_Recommendation
302,21,Male,93.1,163,Obesity,Moderate,Sedentary,158.8,171,77.2,Gluten,8.5,Balanced
712,52,Male,105.4,175,Obesity,Mild,Active,186.7,173,104.1,Peanuts,3.6,Balanced
243,30,Male,60.5,152,Diabetes,Moderate,Sedentary,217.4,110,94.7,Gluten,3.3,Low_Carb
3,32,Male,58.1,164,Hypertension,Mild,Moderate,168.2,144,159.4,Gluten,4.3,Balanced
847,62,Female,84.2,188,Hypertension,Severe,Moderate,242.1,142,74.5,Gluten,7.5,Low_Sodium


In [32]:
train_df.shape

(995, 13)

In [33]:
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
# from catboost import CatBoostClassifier
# 2. LightGBM
from lightgbm import LGBMClassifier
# 3. XGBoost
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,PowerTransformer,RobustScaler,FunctionTransformer,OrdinalEncoder
from sklearn.metrics import confusion_matrix,classification_report,log_loss
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier


In [34]:
X=train_df.drop(columns=['Diet_Recommendation'])
y=train_df['Diet_Recommendation']


In [35]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [49]:
x_train,x_test,y_train,y_test=train_test_split(X,y_encoded,random_state=42,test_size=0.2)

In [37]:
continue_data=[]
binary_data=[]
cat_data=train_df.select_dtypes(include=['object']).columns
num_data=train_df.select_dtypes(include=['int64','float64']).columns
for i in num_data:
    if len(train_df[i].unique())>10:
        continue_data.append(i)
    else:
        binary_data.append(i)

In [38]:
binary_data

[]

In [39]:
def add_calorie_feature(X):

    X = X.copy()

    # ---- BMI calculation ----
    height_m = X["Height_cm"] / 100
    X["BMI"] = X["Weight_kg"] / (height_m ** 2)

    # ---- BMR calculation ----
    BMR = np.where(
        X["Gender"].str.lower() == "male",
        10 * X["Weight_kg"] + 6.25 * X["Height_cm"] - 5 * X["Age"] + 5,
        10 * X["Weight_kg"] + 6.25 * X["Height_cm"] - 5 * X["Age"] - 161
    )

    # ---- activity multiplier ----
    activity_map = {
        "sedentary": 1.2,
        "moderate": 1.55,
        "active": 1.725
    }

    activity_factor = X["Physical_Activity_Level"].str.lower().map(activity_map)

    # ---- final calorie intake ----
    X["Calculated_Calorie_Intake"] = BMR * activity_factor

    return X

In [40]:
num_data

Index(['Age', 'Weight_kg', 'Height_cm', 'Cholesterol_mg/dL',
       'Blood_Pressure_mmHg', 'Glucose_mg/dL', 'Weekly_Exercise_Hours'],
      dtype='object')

In [41]:
preprocess = ColumnTransformer(

    transformers=[

        ('skewed',
         PowerTransformer(),
         ['Cholesterol_mg/dL','Glucose_mg/dL']),

        ('outliers',
         RobustScaler(),
         ['Blood_Pressure_mmHg']),

        ('ord',
         OrdinalEncoder(categories=[
             ['Mild','Moderate','Severe'],
             ['Sedentary','Moderate','Active']
         ]),
         ['Severity','Physical_Activity_Level']),

        ('cat',
         OneHotEncoder(handle_unknown='ignore'),
         ['Gender','Disease_Type', 'Allergies',]),

        ('num',
         StandardScaler(),
         ['Age','Weight_kg','Height_cm','Weekly_Exercise_Hours','BMI', 'Calculated_Calorie_Intake'])

    ]

)

In [42]:
final_prep=Pipeline(steps=[
    ('engineering', FunctionTransformer(add_calorie_feature)),
    ('prep', preprocess),
    ("model", XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42))
])

In [45]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

score = cross_val_score(
    final_prep,
    X,
    y_encoded,
    scoring='f1_macro',
    cv=kfold
)

In [46]:
score.mean()

np.float64(0.7683485586449077)

In [50]:
final_prep.fit(x_train,y_train)

,steps,"[('engineering', ...), ('prep', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function add...002CA76557EB0>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [51]:
y_pred=final_prep.predict(x_test)

In [52]:
from sklearn.metrics import classification_report,accuracy_score

In [53]:
accuracy_score(y_test,y_pred)

0.7386934673366834

In [55]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.76      0.66      0.70        94
           1       1.00      1.00      1.00        48
           2       0.54      0.65      0.59        57

    accuracy                           0.74       199
   macro avg       0.76      0.77      0.76       199
weighted avg       0.75      0.74      0.74       199



In [47]:
import pickle
with open('pipeline.pkl', 'wb') as file:
    pickle.dump(final_prep, file)
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)
X.sample(3)

,Age,Gender,Weight_kg,Height_cm,Disease_Type,Severity,Physical_Activity_Level,Cholesterol_mg/dL,Blood_Pressure_mmHg,Glucose_mg/dL,Allergies,Weekly_Exercise_Hours
912,53,Female,51.6,160,Obesity,Moderate,Moderate,200.5,140,187.5,Gluten,9.9
748,53,Male,77.5,157,Hypertension,Severe,Moderate,151.8,134,163.9,Peanuts,9.5
166,61,Male,104.0,177,Hypertension,Moderate,Active,224.8,151,152.0,Gluten,3.6


In [ ]:
train_df.columns

Index(['Age', 'Gender', 'Weight_kg', 'Height_cm', 'Disease_Type', 'Severity',
       'Physical_Activity_Level', 'Cholesterol_mg/dL', 'Blood_Pressure_mmHg',
       'Glucose_mg/dL', 'Allergies', 'Weekly_Exercise_Hours',
       'Diet_Recommendation'],
      dtype='object')